# Lecture 5. Messy and missing data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aaubs/AQM-2026/blob/main/lecture05/lecture05.ipynb)

Applied Quantitative Methods, Fall 2026. Readings: all earlier readings, with Newbold, Carlson and Thorne, sections
1.1 and 1.2 (nonresponse and missing values), 2.1 (mean, median, quartiles and outliers) and 2.2 (interquartile
range, box plots and the empirical rule).
The two videos on Moodle (data cleaning in pandas, and the three types of missing data) are optional.

This notebook follows the lecture slides. Until now you have worked with a clean file of 200 orders. Today you get the
file as it came out of RetailCorp's two order systems, and we clean it one problem at a time. Each step follows the
same sequence: **look** at the problem, **decide** what to do, **do** it in code, **check** that it worked, and
**log** the decision. At the end we run the confidence interval from Lecture 2 and the warehouse test from Lecture 4
on the raw file and on the cleaned file and compare.

Parts 1, 2 and 4 each have an exercise: you predict a result, run the cell, change one value, and answer a question
about what you see. You do not need to write any code. The last section holds the cleaning steps, the pandas functions
and the Danish terms, so you can use the notebook as a reference for the exam assignment.

Run the setup cell first.

## Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

url = "https://raw.githubusercontent.com/aaubs/AQM-2026/main/data/"
raw = pd.read_csv(url + "orders_export.csv")              # today's file, as exported
sample = pd.read_csv(url + "orders_sample.csv")           # the clean study file from Lectures 1 to 4
population = pd.read_csv(url + "orders_population.csv")   # all 50,000 orders, used only in the demonstration

print(f"Rows in the export: {len(raw)}. Rows in the study file: {len(sample)}.")

## The case

RetailCorp moved to a new order system on 1 April 2026. Orders until 31 March are in the old system, orders from 1
April in the new one. An analyst asked IT for the 200 orders of the study you have used since Lecture 1, and got one
file with the export from both systems: `orders_export.csv`. The study file `orders_sample.csv` is what the analyst
made from it after cleaning. She could also look up single orders in the order system; we only have the export. The
export has three columns that the study file left out: the system, the number of items in the order and the
customer's rating.

**What does the raw export look like, and do the answers from Lectures 2 to 4 change if we skip the cleaning?**

The data description from IT:

| Column | Content |
|---|---|
| `order_id` | Order number |
| `system` | `old` (until 31 March) or `new` (from 1 April) |
| `order_date` | Date of the order |
| `warehouse` | Warehouse that shipped the order: Aarhus or Copenhagen |
| `carrier` | Carrier A or Carrier B |
| `region` | The customer's region |
| `delivery_hours` | Hours from order to delivery |
| `late` | Delivered after more than 48 hours |
| `order_value_dkk` | Value of the order in DKK |
| `n_items` | Number of items in the order |
| `rating` | The customer's rating of the delivery, 1 to 5 stars. The old system records "no answer" as 99 |

## Part 1. A first look

Before any analysis, look at three things: how many rows and columns there are, what type each column has, and which
values occur.

**Types of variables (Newbold 1.2).** A categorical variable puts each unit in a group (warehouse, region). A numerical
variable is a count or a measurement (number of items, delivery hours). pandas has its own names for the types it
stores: `int64` for whole numbers, `float64` for numbers with decimals (and for whole numbers when some values are
missing), and `object` for text. A column of numbers that pandas reads as `object` contains at least one value that
is not a number in Python's format.

In [ ]:
raw.head()

In [ ]:
raw.info()

`info()` shows the number of rows, the number of non-missing values in each column (`Non-Null Count`), and the type.
Which columns should be numbers but are stored as text? Which columns have missing values?

In [ ]:
raw["warehouse"].value_counts()

In [ ]:
raw.describe()

`describe()` only covers the columns pandas reads as numbers. Look at the minimum and maximum of each: are they
possible?

### Exercise 1. Find the problems

**Predict.** Which columns should be numbers? How many different labels should `warehouse` have?

**Run** the cell. It prints the type of each column, the labels in one column, and the number of rows that are exact
copies of an earlier row.

In [ ]:
column = "warehouse"            # Change: set to "late", then to "region"

print(raw.dtypes, "\n")
print(raw[column].value_counts(dropna=False), "\n")
print(f"Rows that are exact copies of an earlier row: {raw.duplicated().sum()}")
print(f"Order numbers that occur more than once: {raw['order_id'].duplicated().sum()}")

**Change** `column` to `"late"` and run again, then to `"region"`.

**Answer.** Write down every problem you can find in the outputs above. For each problem: would Python stop with an
error, or would it give a wrong number without warning?

## Part 2. Cleaning one problem at a time

We keep `raw` as it is and clean a copy, `df`, so we can always compare with the original. Every decision goes into
a cleaning log: the problem, the number of rows affected, the decision, and what would happen if we did not handle it.
A log written while you clean is easier than one written afterward.

In [ ]:
df = raw.copy()
log = []        # one entry per problem: (problem, rows, decision, consequence if not handled)

### Duplicate rows

**Look.** The new system's export for 1 to 7 April was included twice. 9 rows are exact copies of an earlier
row, with the same order number.

In [ ]:
print(df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("order_id").head(4)

**Decide, do, check, log.** The same order cannot have been placed twice with the same number, so we keep the first
copy. `drop_duplicates()` removes rows that are identical in every column.

In [ ]:
print(df.duplicated().sum())                        # look
df = df.drop_duplicates()                           # do
print(len(df), df["order_id"].duplicated().sum())   # check
log.append(("Exact duplicate rows", 9, "Removed (export for 1 to 7 April included twice)",
            "n = 209 instead of 200; nine orders count twice and the standard error is too small"))

### Inconsistent labels

**Look.** The old system writes the warehouse in Danish (Århus, København), the new one in English, and a few labels
were typed by hand with a lower-case letter or a space at the end. To pandas, `"Aarhus"` and `"Aarhus "` are two
different groups.

In [ ]:
print(df["warehouse"].value_counts())

**Do.** First remove spaces at both ends (`str.strip()`) and use lower case (`str.lower()`), so that only the real
differences remain. Then `map` each label to one spelling. **Check:** a label that is not in the dictionary becomes
missing (`NaN`), so count the missing values afterward.

In [ ]:
key = df["warehouse"].str.strip().str.lower()
df["warehouse"] = key.map({"aarhus": "Aarhus",
                           "århus": "Aarhus",
                           "copenhagen": "Copenhagen",
                           "københavn": "Copenhagen"})
print(df["warehouse"].value_counts(dropna=False))
log.append(("Warehouse labels", 93, "Mapped Århus, København and typed variants to Aarhus and Copenhagen",
            "Rows with other spellings drop out of every comparison by warehouse"))

### Numbers stored as text

A column is read as text if a single value in it is not a number in Python's format. The old system writes numbers the
Danish way: a decimal comma (`28,5`), a dot as thousand separator and a unit (`1.582,37 kr.`). Python needs a decimal
point and nothing else (`1582.37`).

**Delivery hours.** Replace the comma with a point, then convert with `astype(float)`.

In [ ]:
print(df["delivery_hours"].head(3).tolist(), df["delivery_hours"].tail(3).tolist())

In [ ]:
df["delivery_hours"] = (df["delivery_hours"]
                        .str.replace(",", ".", regex=False)
                        .astype(float))
print(df["delivery_hours"].dtype, df["delivery_hours"].isna().sum())

In [ ]:
print(df["delivery_hours"].describe().round(3))

**Late: from Ja, Nej, Yes and No to 1 and 0.** `map` turns the four labels into numbers, and `astype(int)` stores them
as whole numbers. **Check** against the definition: an order is late if it took more than 48 hours.

In [ ]:
print(df["late"].value_counts())

In [ ]:
df["late"] = df["late"].map({"Yes": 1, "Ja": 1,
                             "No": 0, "Nej": 0}).astype(int)
print((df["late"] == (df["delivery_hours"] > 48)).all())   # check

**Order value.** Here the two systems use the dot for different things. In the old system `1.582,37 kr.` has a
thousand separator; in the new system `1582.37` has a decimal point. So we convert the old system's rows on their own,
in this order: remove the unit, remove the thousand separator, turn the comma into a point.

In [ ]:
print(df.loc[df["system"] == "old", "order_value_dkk"].head(3).tolist(),
      df.loc[df["system"] == "new", "order_value_dkk"].head(3).tolist())

In [ ]:
old = df["system"] == "old"
text = df["order_value_dkk"].copy()
text.loc[old] = (text.loc[old].str.replace(" kr.", "", regex=False)  # 1 unit
                              .str.replace(".", "", regex=False)     # 2 thousands
                              .str.replace(",", ".", regex=False))   # 3 decimal
df["order_value_dkk"] = text.astype(float)
print(df["order_value_dkk"].dtype, df["order_value_dkk"].isna().sum())
log.append(("Numbers stored as text", 89, "Decimal comma, thousand separator and unit removed in old-system rows",
            "Columns cannot be used for calculations; a forced conversion loses the old-system rows"))
log.append(("Late as Ja, Nej, Yes, No", 200, "Mapped to 1 and 0, checked against delivery_hours > 48",
            "No share of late orders can be computed"))

**Three ways to convert, and what they do with a value that does not fit.** `astype(float)` stops with an error,
which is what you want: you notice the problem. `pd.to_numeric(..., errors="coerce")` turns every value it cannot read
into a missing value, without a warning. Online answers and LLMs often suggest it because the code then runs. On this
file it silently drops all 89 rows from the old system. `astype(int)` stops if the column has missing values,
because `NaN` is a float; keep such columns as `float`.

In [ ]:
try:
    raw["delivery_hours"].astype(float)
except ValueError as error:
    print("astype(float) stopped:", error)
coerced = pd.to_numeric(raw["delivery_hours"], errors="coerce")
print(coerced.isna().sum(), "values became missing")

### Exercise 2. Convert a text column

The cell converts the old system's order values with the three steps above, using `pd.to_numeric(errors="coerce")`.

**Predict.** If you skip step 2 (removing the thousand separator), what happens to the number of values converted,
and to the mean and the median? Will Python warn you?

**Run** the cell.

In [ ]:
remove_thousands = True       # Change: set to False

text_old = raw.loc[raw["system"] == "old", "order_value_dkk"]
text_old = text_old.str.replace(" kr.", "", regex=False)
if remove_thousands:
    text_old = text_old.str.replace(".", "", regex=False)
text_old = text_old.str.replace(",", ".", regex=False)
values = pd.to_numeric(text_old, errors="coerce")

print(f"Converted: {values.count()}, missing after conversion: {values.isna().sum()}")
print(f"Mean: {values.mean():.2f} DKK, median: {values.median():.2f} DKK")

**Change** `remove_thousands` to `False` and run again. Was your prediction right?

**Answer.** Which orders were lost, and why does the mean fall more than the median? What should you always count
before and after a conversion?

### Codes that stand for missing

**Look.** The data description says that the old system records "no answer" as 99. To pandas, 99 is just a number,
so it enters the mean (Newbold 1.2: missing values need a special code, and unless they are handled the output is
wrong).

In [ ]:
print(df["rating"].value_counts(dropna=False).sort_index())

In [ ]:
print(f"Mean with 99: {df['rating'].mean():.2f}")
df["rating"] = df["rating"].replace(99, np.nan)
print(f"Mean after: {df['rating'].mean():.2f}, ratings: {df['rating'].count()}")

In [ ]:
log.append(("Rating coded 99 (no answer, old system)", 20, "Set to missing",
            "Mean rating 14.84 on a 1 to 5 scale instead of 4.26"))

### Impossible values

**Look.** An order has at least one item. We cannot know the right number, so we set the impossible values to
missing. `mask(condition)` hides the values where the condition is true.

In [ ]:
print(df.loc[df["n_items"] <= 0, ["order_id", "n_items"]])
df["n_items"] = df["n_items"].mask(df["n_items"] <= 0)

In [ ]:
log.append(("Number of items 0 or negative", 3, "Set to missing", "Mean and minimum of n_items are wrong"))

### Real values and errors

Order values are skewed: most orders are small and a few are large. A large value is not an error in itself. Newbold
(2.1) says that when there are outliers, look for the cause first; one cause can be an error in data entry. Outliers
move the mean but hardly the median.

In [ ]:
print(df["order_value_dkk"].describe())
print(df["order_value_dkk"].sort_values(ascending=False).head(5))

**Two rules that flag values.** Newbold calls a value more than 3 standard deviations from the mean an outlier (the
empirical rule, 2.2, meant for roughly normal data). The box plot in Python flags values more than 1.5 × IQR above
the third quartile. On skewed data the second rule flags many real orders, so a flag is a reason to look at the order.
The 3 s rule flags only one order here partly because that order inflates s; without it, the rule flags 4 orders.

In [ ]:
x = df["order_value_dkk"]
q1, q3 = x.quantile(0.25), x.quantile(0.75)
iqr = q3 - q1
print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f}, IQR = {iqr:.2f}")
print(f"Above Q3 + 1.5 IQR = {q3 + 1.5 * iqr:.1f}: {(x > q3 + 1.5 * iqr).sum()} orders")
print(f"Above mean + 3 s = {x.mean() + 3 * x.std():.1f}: {(x > x.mean() + 3 * x.std()).sum()} order")

plt.boxplot(x.dropna())
plt.ylabel("Order value (DKK)")
plt.show()

**Look for the cause.** Value per item puts each order on the same footing. An order of 15,823.70 DKK with 10 items
is about 1,580 DKK per item; no other order comes above 500 DKK per item. The largest real order, 3,274.42 DKK, has 20
items. The most likely cause is an extra zero typed in the new system. We only have the export, so we set the value
to missing. The analyst, who could look the order up, divided by 10 instead, which is also defensible; that is why the
study file has 1,582.37 and a mean order value of 569.19.

In [ ]:
per_item = df["order_value_dkk"] / df["n_items"]
top = df.assign(per_item=per_item).sort_values("per_item", ascending=False)
print(top[["order_id", "order_value_dkk", "n_items", "per_item"]].head(4).round(2))

In [ ]:
v = df["order_value_dkk"]
print(f"With:    mean {v.mean():.2f}, median {v.median():.2f}")
df["order_value_dkk"] = v.mask(per_item > 1000)
v = df["order_value_dkk"]
print(f"Without: mean {v.mean():.2f}, median {v.median():.2f}")

In [ ]:
log.append(("Order value with an extra zero", 1, "Set to missing (about 1,580 DKK per item; others below 500)",
            "Mean order value 640.40 instead of 564.10; the median hardly moves"))

### The cleaning log

The table you will make in section 2 of the exam assignment. The last check: the cleaned delivery times, warehouses
and late flags are the study file from Lectures 1 to 4.

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(log, columns=["Problem", "Rows", "Decision", "Consequence if not handled"])

In [ ]:
df = df.reset_index(drop=True)
same = all((df[c].values == sample[c].values).all() for c in ["order_id", "warehouse", "delivery_hours", "late"])
print("Cleaned file equals the study file in these columns:", same)

### Keeping your work in Colab

* **Save the notebook.** A notebook opened from the course link is not saved anywhere. Use *File > Save a copy in
Drive*, and share that copy with your group.
* **Colab forgets files.** Files you write go to a temporary folder, `/content`, which is deleted when the session
ends.
* **The cleaning code is the record.** *Runtime > Run all* reads the raw file and redoes every step in a few seconds.
* **For the exam.** The notebook you hand in loads the raw file from the course repository and does the cleaning
itself. A saved cleaned file only saves time while you work.

Save the cleaned data and read it back to check that nothing changed. `pd.testing.assert_frame_equal` stops with an
error if the two differ, and allows for rounding in the last decimals of a saved number. `index=False` leaves out the row numbers;
without it they come back as an extra column, `Unnamed: 0`. Column types are read again from the text, so a column
with missing values comes back as `float`.

In [ ]:
df.to_csv("orders_clean.csv", index=False)
check = pd.read_csv("orders_clean.csv")
pd.testing.assert_frame_equal(check, df)   # stops if they differ
print("Saved file matches:", check.shape)

The file above sits in Colab's temporary folder. There are two ways to keep it between sessions. Both cells work only
in Colab.

**In your Google Drive.** The file stays between sessions, and your group can share the folder. `drive.mount` asks for
permission to your Drive the first time in each session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")    # asks for access
folder = "/content/drive/MyDrive/AQM"
os.makedirs(folder, exist_ok=True)
df.to_csv(folder + "/orders_clean.csv", index=False)

# next time, after drive.mount:
# df = pd.read_csv(folder + "/orders_clean.csv")

**On your computer.** `files.download` sends the file to your Downloads folder. Next time, upload it to Colab before you
read it.

In [ ]:
df.to_csv("orders_clean.csv", index=False)
from google.colab import files
files.download("orders_clean.csv")

# next time: upload the file with the folder
# icon on the left, or files.upload(), then
# df = pd.read_csv("orders_clean.csv")

## Part 3. Missing data

After cleaning, some cells are empty. What matters is why they are empty, because that decides whether the rows we
still have describe all orders. There are three cases, from least to most serious.

* **Missing completely at random.** Whether a value is missing has nothing to do with the order or the customer.
Example: four orders where the region field is empty, with no pattern we can see.
* **Missing at random.** A value is missing more often in a group we can see in the data. Example: ratings are
missing more often after late orders (44% against 18%). Within each group, the ratings
we have can still describe that group.
* **Missing not at random.** Whether a value is missing depends on the missing value itself. Example: customers who
would give a low rating skip the rating, also when the order was on time. The data cannot show this, because the
ratings we would need are the ones that are missing. Newbold (1.1) describes the same problem for nonresponse: the
results describe the people who were willing to respond.

In [ ]:
print(df.isna().sum())

In [ ]:
print("Share of ratings missing, by late:")
print(df["rating"].isna().groupby(df["late"]).mean().round(3), "\n")
print(df.groupby("late")["rating"].agg(["mean", "count", "size"]).round(2))

Late orders are 16 of 200 orders (8%), but only 9 of the 159 ratings
(5.7%), and they get much lower ratings. So the average of the ratings we have,
4.26, leans toward on-time orders and is probably a little too high as a picture of all orders. If the
ratings are missing at random, weighting the two groups by their share of all orders gives
4.22. If they are missing not at random, no calculation on these data can correct it.

### Missing values in Python and in Newbold

Newbold (1.2) says missing values need a special code. pandas uses `NaN` (not a number) and handles it quietly:

* `read_csv` turns empty cells into `NaN`, but not codes such as 99 or −1. Those you must replace yourself.
* `.mean()`, `.std()` and `.sum()` skip `NaN` without a warning. `.count()` counts the values that are there,
`len()` counts all rows. Use `.count()`, or `len()` after `dropna()`, as n.

In [ ]:
r = df["rating"]
print(f"len: {len(r)}, count: {r.count()}")
print(f"mean(): {r.mean():.3f}")
print(f"sum() / len(): {r.sum() / len(r):.3f}")

`mean()` skips the 41 missing ratings. `sum() / len()` divides by all 200 rows, as if every missing rating were 0.

### Dropping rows or filling in values

* **Drop** the rows without a value for the analysis that needs it (Newbold drops a pair with a missing value in
section 8.1). n gets smaller and the standard error is honest. If the values are missing at random or not at random,
the result describes the rows that are left.
* **Fill in** a value, for example the mean. n stays the same, but every filled value sits exactly at the mean, so the
standard deviation shrinks and the interval becomes too narrow. Filling also does not fix the lean toward on-time
orders.

In [ ]:
dropped = df["rating"].dropna()
filled = df["rating"].fillna(df["rating"].mean())

for name, x in (("dropped", dropped), ("filled with the mean", filled)):
    n, m, s = x.count(), x.mean(), x.std(ddof=1)
    me = stats.t.ppf(0.975, df=n - 1) * s / np.sqrt(n)
    print(f"{name:21} n = {n}, mean = {m:.3f}, s = {s:.3f}, 95% interval [{m - me:.3f}, {m + me:.3f}]")

### Demonstration. Filling with the mean makes the interval too narrow

Because the population of 50,000 orders is simulated, we know the true mean delivery time. The cell draws 1,000 samples
of 200 orders, removes 60 delivery times (30%) at random, and builds a 95% interval in two ways: from the 140 values
that are left, and after filling the 60 gaps with the mean of the 140. It counts how often each interval contains the
true mean.

In [ ]:
mu = population["delivery_hours"].mean()
hours = population["delivery_hours"].to_numpy()
rng = np.random.default_rng(5)
reps, n, n_miss = 1000, 200, 60
covered_drop, covered_fill = 0, 0
for i in range(reps):
    x = rng.choice(hours, size=n, replace=False)
    kept = x[n_miss:]                                          # 60 values removed at random
    filled = np.concatenate([kept, np.full(n_miss, kept.mean())])
    for data, method in ((kept, "drop"), (filled, "fill")):
        m, s, k = data.mean(), data.std(ddof=1), len(data)
        me = stats.t.ppf(0.975, df=k - 1) * s / np.sqrt(k)
        if m - me <= mu <= m + me:
            if method == "drop":
                covered_drop += 1
            else:
                covered_fill += 1
print(f"True mean: {mu:.2f} hours")
print(f"Intervals that contain it, dropping the missing values: {covered_drop / reps:.1%}")
print(f"Intervals that contain it, filling with the mean:      {covered_fill / reps:.1%}")

In [ ]:
log.append(("Rating missing (not given)", int(df["rating"].isna().sum()),
            "Kept as missing; analyses of rating use the rows with a rating",
            "Missing more often after late orders, so the mean rating leans toward on-time orders"))
log.append(("Region empty", int(df["region"].isna().sum()), "Kept as missing", "Four orders drop out of comparisons by region"))
pd.DataFrame(log, columns=["Problem", "Rows", "Decision", "Consequence if not handled"])

## Part 4. Back to Lectures 2 to 4

### Exploratory analysis of the cleaned file

Exploratory analysis means getting to know the data before testing: the center and spread of each variable, the
shape of the distributions, differences between groups and correlations (Newbold chapters 1 and 2).

In [ ]:
cols = ["delivery_hours", "order_value_dkk", "n_items", "rating"]
df[cols].describe().round(2)

In [ ]:
x = df["order_value_dkk"].dropna()
plt.hist(x, bins=20, color="#31A9C1")
plt.axvline(x.mean(), color="#DF8E2E", label=f"mean {x.mean():.0f}")
plt.axvline(x.median(), color="#211A52", label=f"median {x.median():.0f}")
plt.xlabel("Order value (DKK)")
plt.legend()
plt.show()
print(f"Skewness: {stats.skew(x):.2f}")

In [ ]:
print(df.groupby("warehouse")["delivery_hours"].agg(["count", "mean", "std"]).round(3), "\n")
print(df[cols].corr().round(2))

The rating falls with delivery time (correlation −0.53), and order value goes with the number of items (0.82).

### The interval from Lecture 2, on the raw file and on the cleaned file

What is the average delivery time? The raw file "forced to run" means: convert with `pd.to_numeric(errors="coerce")`,
and leave labels and duplicates as they are.

In [ ]:
forced = raw.copy()
forced["delivery_hours"] = pd.to_numeric(forced["delivery_hours"],
                                         errors="coerce")
for name, data in (("forced", forced), ("cleaned", df)):
    x = data["delivery_hours"].dropna()
    n, x_bar, s = len(x), x.mean(), x.std(ddof=1)
    me = stats.t.ppf(0.975, df=n - 1) * s / np.sqrt(n)
    print(f"{name:8} n = {n}, x_bar = {x_bar:.3f}, s = {s:.3f}, "
          f"95% interval [{x_bar - me:.2f}, {x_bar + me:.2f}]")

As a check, scipy's function for the t interval gives the same limits for the cleaned file (the last one in the loop).

In [ ]:
lo, hi = stats.t.interval(0.95, n - 1, loc=x_bar, scale=s / np.sqrt(n))
print(f"[{lo:.2f}, {hi:.2f}]")

The cleaned file gives the Lecture 2 interval, [35.65, 37.92] hours. The forced version uses 120 delivery
times: the 89 old-system values were lost without a warning, and nine orders count twice. Its interval is
wider, and it describes April to June only.

### Exercise 3. The warehouse test on the raw and the cleaned file

The cell runs the one-sided Welch test from Lecture 4 (do Copenhagen orders take longer?) on the forced or on the
cleaned file.

**Predict.** In Lecture 4 the gap was 2.34 hours (t = 1.975, p = 0.025). On the forced file, will the gap be larger,
smaller or about the same? How many orders will be compared?

**Run** the cell.

In [ ]:
route = "forced"          # Change: set to "cleaned"

data = forced if route == "forced" else df
x1 = data.loc[data["warehouse"] == "Copenhagen", "delivery_hours"].dropna()
x2 = data.loc[data["warehouse"] == "Aarhus", "delivery_hours"].dropna()
n1, n2 = len(x1), len(x2)
v1, v2 = x1.var(ddof=1) / n1, x2.var(ddof=1) / n2
t_stat = (x1.mean() - x2.mean()) / np.sqrt(v1 + v2)
df_w = (v1 + v2)**2 / (v1**2 / (n1 - 1) + v2**2 / (n2 - 1))
p = stats.t.sf(t_stat, df=df_w)

print(f"{route}: Copenhagen n = {n1}, Aarhus n = {n2}, gap = {x1.mean() - x2.mean():.2f} hours")
print(f"t = {t_stat:.3f}, df = {df_w:.2f}, p = {p:.4f}")
print("Orders compared, by system:", data.loc[x1.index.union(x2.index), "system"].value_counts().to_dict())

**Change** `route` to `"cleaned"` and run again.

**Answer.** Why is the gap so different? Which orders were left out of the forced version, and were they left out at
random?

## Reference

### The cleaning sequence

For every problem: **look** (count the rows affected), **decide** (and be able to say why), **do** (in code, on a copy
of the raw data), **check** (count again; no unexpected missing values), **log** (problem, rows, decision,
consequence).

| Problem | Look | Do | Check |
|---|---|---|---|
| Duplicate rows | `df.duplicated().sum()` | `df.drop_duplicates()` | `len(df)`, ids unique |
| Inconsistent labels | `df["col"].value_counts()` | `.str.strip().str.lower()`, then `.map({...})` | `value_counts(dropna=False)`: no `NaN` |
| Code for missing | the data description; `value_counts()` | `.replace(99, np.nan)` | the mean is within the scale |
| Impossible value | `describe()`: minimum and maximum | `.mask(condition)` | `isna().sum()` |
| Number stored as text | `info()`: type `object` | `.str.replace(...)`, then `.astype(float)` | type `float64`, no new `NaN` |
| Yes/No as text | `value_counts()` | `.map({"Yes": 1, "No": 0}).astype(int)` | counts before and after agree |
| Unusual value | `describe()`, sorting, a box plot, a ratio such as value per item | `.mask(condition)` if it is an error | mean and median before and after |
| Missing values | `isna().sum()`; share missing by group | analyses use the rows with a value | report n |

### The raw file against the cleaned file

| | Problem not handled | Cleaned |
|---|---|---|
| Delivery times used (forced to run) | 120 (April to June, nine twice) | 200 |
| 95% interval for the mean | [35.12, 38.31] | [35.65, 37.92] |
| Copenhagen against Aarhus | 50 and 66 orders, gap 5.16 hours, t = 3.249, p = 0.0008 | 80 and 120 orders, gap 2.34 hours, t = 1.975, p = 0.025 |
| Mean rating | 14.84 with 99 kept | 4.26 (159 ratings) |
| Mean order value | 640.40 with the extra zero | 564.10 (median 427.14) |

### pandas functions and their defaults

| Function | Watch out for |
|---|---|
| `pd.read_csv` | Empty cells become `NaN`; codes such as 99 do not. A column with one non-number becomes text (`object`). |
| `.astype(float)`, `.astype(int)` | Stop with an error if a value does not fit. `int` cannot hold `NaN`. |
| `pd.to_numeric(x, errors="coerce")` | Turns every value it cannot read into `NaN`, without a warning. Count the missing values before and after. |
| `.str.replace(old, new, regex=False)` | With `regex=True` a dot means "any character". Write `regex=False` for plain text. |
| `.map(dictionary)` | A value that is not in the dictionary becomes `NaN`. |
| `.mean()`, `.std()`, `.sum()` | Skip `NaN` without a warning. `.count()` gives the number of values used. |
| `.fillna(value)` | Filling with the mean shrinks the standard deviation. |
| `df.to_csv(path, index=False)` | Without `index=False` the row numbers are saved as an extra column. In Colab, a file is kept between sessions only in your Drive or after `files.download`. |
| `.quantile(0.25)` | Uses a slightly different rule for quartiles than Newbold; the numbers can differ a little. |
| `plt.boxplot` | Whiskers reach the most extreme value within 1.5 × IQR of the box, and values beyond are drawn as points. In Newbold the whiskers go to the minimum and maximum. |

For any function, and for code an LLM writes: check what it does with values it cannot read before you use the result.

### Danish terms

| English | Danish |
|---|---|
| data cleaning | datarensning |
| missing value | manglende værdi |
| duplicate | dublet |
| outlier | outlier, ekstrem observation |
| imputation (filling in values) | imputering |
| missing completely at random | mangler helt tilfældigt (MCAR) |
| missing at random | mangler tilfældigt (MAR) |
| missing not at random | mangler ikke tilfældigt (MNAR) |
| nonresponse | manglende besvarelse, bortfald |
| interquartile range | interkvartilafstand |
| box plot | boksplot |